In [3]:
# 2. Import
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import SMAPE
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning import seed_everything

In [4]:
# 3. Đọc dữ liệu
ankhe_df = pd.read_csv("AnKhe.csv")
kanak_df = pd.read_csv("KaNak.csv")
rain_df = pd.read_csv("SoLieuMua_AnKhe_Kanak.csv")

ankhe_df = ankhe_df.rename(columns={"luuluongdenho": "luu_luong_den_ho"})
kanak_df = kanak_df.rename(columns={"luu_luong_xa": "luu_luong_xa"})

# Chuyển datetime
ankhe_df["datetime"] = pd.to_datetime(ankhe_df["gio"].astype(str) + " " + ankhe_df["ngay"], format="%H %d/%m/%Y", errors='coerce')
kanak_df["datetime"] = pd.to_datetime(kanak_df["gio"].astype(str) + " " + kanak_df["ngay"], format="%H %d/%m/%Y", errors='coerce')
rain_df["datetime"] = pd.to_datetime(rain_df["datetime"], errors='coerce')

# Lọc & rename
ankhe_df = ankhe_df[["datetime", "luu_luong_den_ho"]].rename(columns={"luu_luong_den_ho": "ankhe_inflow"})
kanak_df = kanak_df[["datetime", "luu_luong_xa"]].rename(columns={"luu_luong_xa": "kanak_outflow"})

# Gộp
df = pd.merge(ankhe_df, kanak_df, on="datetime", how="inner")
df = pd.merge(df, rain_df, on="datetime", how="inner")

# Tổng lượng mưa từ 3 trạm
df["rain"] = df[["hokanak135928", "hoankhe135931", "trammuavinhthuan82897"]].sum(axis=1)

# Tạo cột chỉ số thời gian & group
df = df.sort_values("datetime").reset_index(drop=True)
df["time_idx"] = np.arange(len(df))
df["group"] = "ankhe_kanak"

In [5]:
# 4. Chia train/test
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]


In [6]:
# 5. Dataset
seq_len = 24
pred_len = 1

training = TimeSeriesDataSet(
    train,
    time_idx="time_idx",
    target="ankhe_inflow",
    group_ids=["group"],
    max_encoder_length=seq_len,
    max_prediction_length=pred_len,
    static_categoricals=["group"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=["ankhe_inflow", "kanak_outflow", "rain"],
)

validation = TimeSeriesDataSet.from_dataset(training, test, predict=True, stop_randomization=True)

# 6. Dataloader
batch_size = 64
train_loader = training.to_dataloader(train=True, batch_size=batch_size)
val_loader = validation.to_dataloader(train=False, batch_size=batch_size)

In [10]:
# 7. Huấn luyện (Alternative approach)
seed_everything(42)

# Create model
tft = TemporalFusionTransformer.from_dataset(
    training, 
    learning_rate=1e-3, 
    hidden_size=16, 
    attention_head_size=1, 
    dropout=0.1, 
    loss=SMAPE()
)

# Train using pytorch-forecasting's method
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, verbose=True, mode="min")],
    enable_progress_bar=True
)

# This should work
trainer.fit(tft, train_loader, val_loader)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


TypeError: `model` must be a `LightningModule` or `torch._dynamo.OptimizedModule`, got `TemporalFusionTransformer`

In [ ]:
# 8. Dự đoán & đánh giá
# Get predictions using pytorch-forecasting method
predictions = tft.predict(val_loader, mode="prediction", return_index=True)

# Extract actual values properly
actuals = torch.cat([y[0] for x, y in iter(val_loader)])

# Convert to numpy for sklearn metrics
if isinstance(predictions, torch.Tensor):
    predictions_np = predictions.cpu().numpy()
else:
    predictions_np = predictions

actuals_np = actuals.cpu().numpy()

# Calculate metrics
mae = mean_absolute_error(actuals_np, predictions_np)
rmse = np.sqrt(mean_squared_error(actuals_np, predictions_np))
r2 = r2_score(actuals_np, predictions_np)

print(f"\nModel Evaluation:\nMAE = {mae:.4f}, RMSE = {rmse:.4f}, R2 = {r2:.4f}")

In [1]:
# 2. Import
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import SMAPE
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning import seed_everything

# 3. Đọc dữ liệu
ankhe_df = pd.read_csv("AnKhe.csv")
kanak_df = pd.read_csv("KaNak.csv")
rain_df = pd.read_csv("SoLieuMua_AnKhe_Kanak.csv")
ankhe_df = ankhe_df.rename(columns={"luuluongdenho": "luu_luong_den_ho"})
kanak_df = kanak_df.rename(columns={"luu_luong_xa": "luu_luong_xa"})

# Chuyển datetime
ankhe_df["datetime"] = pd.to_datetime(ankhe_df["gio"].astype(str) + " " + ankhe_df["ngay"], format="%H %d/%m/%Y", errors='coerce')
kanak_df["datetime"] = pd.to_datetime(kanak_df["gio"].astype(str) + " " + kanak_df["ngay"], format="%H %d/%m/%Y", errors='coerce')
rain_df["datetime"] = pd.to_datetime(rain_df["datetime"], errors='coerce')

# Lọc & rename
ankhe_df = ankhe_df[["datetime", "luu_luong_den_ho"]].rename(columns={"luu_luong_den_ho": "ankhe_inflow"})
kanak_df = kanak_df[["datetime", "luu_luong_xa"]].rename(columns={"luu_luong_xa": "kanak_outflow"})

# Gộp
df = pd.merge(ankhe_df, kanak_df, on="datetime", how="inner")
df = pd.merge(df, rain_df, on="datetime", how="inner")

# Tổng lượng mưa từ 3 trạm
df["rain"] = df[["hokanak135928", "hoankhe135931", "trammuavinhthuan82897"]].sum(axis=1)

# Tạo cột chỉ số thời gian & group
df = df.sort_values("datetime").reset_index(drop=True)
df["time_idx"] = np.arange(len(df))
df["group"] = "ankhe_kanak"

# 4. Chia train/test
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

# 5. Dataset
seq_len = 24
pred_len = 1
training = TimeSeriesDataSet(
    train,
    time_idx="time_idx",
    target="ankhe_inflow",
    group_ids=["group"],
    max_encoder_length=seq_len,
    max_prediction_length=pred_len,
    static_categoricals=["group"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=["ankhe_inflow", "kanak_outflow", "rain"],
)
validation = TimeSeriesDataSet.from_dataset(training, test, predict=True, stop_randomization=True)

# 6. Dataloader
batch_size = 64
train_loader = training.to_dataloader(train=True, batch_size=batch_size)
val_loader = validation.to_dataloader(train=False, batch_size=batch_size)

# 7. Huấn luyện
seed_everything(42)

# Create model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    loss=SMAPE()
)

# Train using pytorch-forecasting's method
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, verbose=True, mode="min")],
    enable_progress_bar=True
)

# Training
trainer.fit(tft, train_loader, val_loader)

# 8. Dự đoán & đánh giá - VERSION ĐÃ SỬA
try:
    # Method 1: Sử dụng predict với return_index=False
    print("Đang thực hiện dự đoán...")
    predictions = tft.predict(val_loader, mode="prediction", return_index=False)
    
    # Lấy actual values từ validation loader
    actuals_list = []
    for batch_idx, (x, y) in enumerate(val_loader):
        if isinstance(y, (list, tuple)):
            # y có thể là tuple (target, weight) hoặc list
            actual_batch = y[0] 
        else:
            actual_batch = y
        actuals_list.append(actual_batch)
    
    actuals = torch.cat(actuals_list, dim=0)
    
    # Xử lý predictions
    if isinstance(predictions, dict):
        # Nếu predictions là dictionary, lấy key chính
        if 'prediction' in predictions:
            pred_values = predictions['prediction']
        else:
            # Lấy key đầu tiên có chứa tensor
            pred_values = next(v for v in predictions.values() if isinstance(v, torch.Tensor))
    elif isinstance(predictions, (list, tuple)):
        pred_values = predictions[0] if len(predictions) > 0 else predictions
    else:
        pred_values = predictions
    
    # Flatten nếu cần thiết
    if len(pred_values.shape) > 1 and pred_values.shape[-1] == 1:
        pred_values = pred_values.squeeze(-1)
    if len(actuals.shape) > 1 and actuals.shape[-1] == 1:
        actuals = actuals.squeeze(-1)
    
    # Convert to numpy
    predictions_np = pred_values.detach().cpu().numpy().flatten()
    actuals_np = actuals.detach().cpu().numpy().flatten()
    
    # Đảm bảo cùng độ dài
    min_len = min(len(predictions_np), len(actuals_np))
    predictions_np = predictions_np[:min_len]
    actuals_np = actuals_np[:min_len]
    
    print(f"Shape predictions: {predictions_np.shape}")
    print(f"Shape actuals: {actuals_np.shape}")
    print(f"Sample predictions: {predictions_np[:5]}")
    print(f"Sample actuals: {actuals_np[:5]}")
    
except Exception as e:
    print(f"Method 1 failed: {e}")
    print("Trying Method 2...")
    
    # Method 2: Manual prediction loop
    tft.eval()
    all_predictions = []
    all_actuals = []
    
    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(val_loader):
            # Forward pass
            pred = tft(x)
            
            # Extract predictions
            if isinstance(pred, dict):
                pred_tensor = pred.get('prediction', pred.get('output', list(pred.values())[0]))
            else:
                pred_tensor = pred
                
            # Extract actuals
            if isinstance(y, (list, tuple)):
                actual_tensor = y[0]
            else:
                actual_tensor = y
            
            all_predictions.append(pred_tensor.cpu())
            all_actuals.append(actual_tensor.cpu())
    
    # Concatenate all batches
    predictions = torch.cat(all_predictions, dim=0)
    actuals = torch.cat(all_actuals, dim=0)
    
    # Flatten
    predictions_np = predictions.numpy().flatten()
    actuals_np = actuals.numpy().flatten()

# Calculate metrics
mae = mean_absolute_error(actuals_np, predictions_np)
rmse = np.sqrt(mean_squared_error(actuals_np, predictions_np))
r2 = r2_score(actuals_np, predictions_np)

print(f"\n=== Model Evaluation ===")
print(f"MAE = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"R2 = {r2:.4f}")

# Optional: Tạo biểu đồ so sánh
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 6))
    
    # Plot first 100 points for clarity
    n_points = min(100, len(actuals_np))
    x_axis = range(n_points)
    
    plt.plot(x_axis, actuals_np[:n_points], label='Actual', color='blue', alpha=0.7)
    plt.plot(x_axis, predictions_np[:n_points], label='Predicted', color='red', alpha=0.7)
    
    plt.xlabel('Time Points')
    plt.ylabel('Ankhe Inflow')
    plt.title('Actual vs Predicted Values')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("Matplotlib not available for plotting")
except Exception as e:
    print(f"Plotting failed: {e}")

print("\n=== Completed Successfully ===")

c:\Users\LinhNguyenKhanh\AppData\Local\Programs\Python\Python39\lib\site-packages\pytorch_forecasting\models\base_model.py:27: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Seed set to 42
c:\Users\LinhNguyenKhanh\AppData\Local\Programs\Python\Python39\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\LinhNguyenKhanh\AppData\Local\Programs\Python\Python39\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: False, used: False
TPU available: False, usi

TypeError: `model` must be a `LightningModule` or `torch._dynamo.OptimizedModule`, got `TemporalFusionTransformer`